In [1]:
import pandas as pd
import altair as alt

In [2]:
%load_ext jupyter_black

In [3]:
crime = pd.read_csv("crime.csv").sort_values("MONTH", ascending=False)

display(crime)

,INCIDENT_NUMBER,OFFENSE_CODE,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,REPORTING_AREA,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,UCR_PART,STREET,Lat,Long,Location
12882,I172102098,616,Larceny,LARCENY THEFT OF BICYCLE,A7,28,NaN,12/9/17 11:15,2017,12,Saturday,11,Part One,CENTRAL SQ,42.374211,-71.039185,"(42.37421104, -71.03918485)"
12798,I172102576,351,Robbery,ROBBERY - BANK,C6,222,NaN,12/11/17 11:31,2017,12,Monday,11,Part One,E BROADWAY,42.335617,-71.036419,"(42.33561661, -71.03641895)"
12807,I172102509,413,Aggravated Assault,ASSAULT - AGGRAVATED - BATTERY,C11,460,NaN,12/11/17 7:15,2017,12,Monday,7,Part One,STRATHCONA RD,42.305211,-71.079301,"(42.30521091, -71.07930130)"
12806,I172102511,614,Larceny From Motor Vehicle,LARCENY THEFT FROM MV - NON-ACCESSORY,A1,125,NaN,12/8/17 10:00,2017,12,Friday,10,Part One,WINCHESTER ST,42.349406,-71.069347,"(42.34940611, -71.06934672)"
12805,I172102528,615,Larceny From Motor Vehicle,LARCENY THEFT OF MV PARTS & ACCESSORIES,E5,822,NaN,12/11/17 9:25,2017,12,Monday,9,Part One,WASHINGTON ST,42.263120,-71.155085,"(42.26312000, -71.15508544)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10613,I182007744,706,Auto Theft,AUTO THEFT - MOTORCYCLE / SCOOTER,C11,352,NaN,1/28/18 23:30,2018,1,Sunday,23,Part One,ARCADIA ST,42.301415,-71.062878,"(42.30141461, -71.06287775)"
10614,I182007742,301,Robbery,ROBBERY - STREET,D4,270,NaN,1/28/18 0:00,2018,1,Sunday,0,Part One,COLUMBUS AVE,42.313701,-71.095512,"(42.31370136, -71.09551192)"
10615,I182007741,619,Larceny,LARCENY ALL OTHERS,C11,345,NaN,1/29/18 18:06,2018,1,Monday,18,Part One,RIDGEWOOD ST,42.302094,-71.065752,"(42.30209443, -71.06575183)"
10616,I182007739,617,Larceny,LARCENY THEFT FROM BUILDING,B3,416,NaN,1/29/18 20:12,2018,1,Monday,20,Part One,CODMAN HILL AVE,42.279219,-71.073864,"(42.27921860, -71.07386433)"


In [4]:
# OCCURRED_ON_DATE is a string like "12/9/17 11:15"
crime["OCCURRED_ON_DATE"] = pd.to_datetime(
    crime["OCCURRED_ON_DATE"], format="%m/%d/%y %H:%M", errors="coerce"
)
crime = crime.dropna(subset=["OCCURRED_ON_DATE"])

# Keep one school year: Sep 1, 2017 - Aug 31, 2018
START, END = "2017-09-01", "2018-08-31"
crime = crime[crime["OCCURRED_ON_DATE"].between(START, f"{END} 23:59:59")].copy()

TYPE = "OFFENSE_CODE_GROUP"
crime["date"] = crime["OCCURRED_ON_DATE"].dt.floor("D")
crime["hour"] = crime["HOUR"].astype(int)
crime["dow"] = crime["DAY_OF_WEEK"].str.strip()

types = ["All"] + sorted(crime[TYPE].dropna().unique())

In [5]:
def count_grid(df, keys, levels):
    """Count incidents per type x keys, including an 'All' type and zero-count cells."""
    both = pd.concat([df.assign(type="All"), df.assign(type=df[TYPE])])
    # Unique incident numbers, so one incident logged with two offense codes
    # (e.g. I142025834-00) counts once
    counts = both.groupby(["type", *keys])["INCIDENT_NUMBER"].nunique()
    idx = pd.MultiIndex.from_product([types, *levels], names=["type", *keys])
    return counts.reindex(idx, fill_value=0).rename("incidents").reset_index()

In [10]:
# %% Pre-aggregate (keeps rows well under Altair's 5,000-row limit)
dates = pd.date_range(START, END, freq="D")
cal = count_grid(crime, ["date"], [dates])
cal["month"] = cal["date"].dt.strftime("%b")
cal["day"] = cal["date"].dt.day
cal["label"] = cal["date"].dt.strftime("%a, %b %d, %Y")
cal = cal.drop(columns="date")

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
week = count_grid(crime, ["dow", "hour"], [days, range(24)])

# %% Charts
type_param = alt.param(
    name="crime_type",
    value="All",
    bind=alt.binding_select(options=types, name="Crime type: "),
)
# School-year order: Sep 2017 -> Aug 2018
months = [
    "Sep",
    "Oct",
    "Nov",
    "Dec",
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
]

# Calendar: which dates have the most incidents
calendar = (
    alt.Chart(cal)
    .transform_filter(alt.datum.type == type_param)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "month:O",
            sort=months,
            title=None,
            axis=alt.Axis(orient="top", labelAngle=0),
        ),
        y=alt.Y("day:O", title=None),
        color=alt.Color(
            "incidents:Q",
            scale=alt.Scale(scheme="purples"),
            legend=alt.Legend(title="Incidents"),
        ),
        tooltip=[
            alt.Tooltip("label:N", title="Date"),
            alt.Tooltip("incidents:Q", title="Incidents"),
        ],
    )
    .properties(
        width=540,
        height=500,
        title="Boston Part One incidents by date, Sep 2017 - Aug 2018",
    )
)

# Day of week x hour: when during the week is riskiest
weekly = (
    alt.Chart(week)
    .transform_filter(alt.datum.type == type_param)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "hour:O", title="Hour of day (0 = midnight)", axis=alt.Axis(labelAngle=0)
        ),
        y=alt.Y("dow:O", sort=days, title=None),
        color=alt.Color(
            "incidents:Q",
            scale=alt.Scale(scheme="purples"),
            legend=alt.Legend(title="Incidents"),
        ),
        tooltip=[
            alt.Tooltip("dow:N", title="Day"),
            alt.Tooltip("hour:O", title="Hour"),
            alt.Tooltip("incidents:Q", title="Incidents"),
        ],
    )
    .properties(
        width=540,
        height=180,
        title="Incidents by day of week and hour, Sep 2017 - Aug 2018",
    )
)

# One dropdown drives both charts
dashboard = (
    alt.vconcat(calendar, weekly)
    .add_params(type_param)
    .resolve_scale(color="independent")
)
dashboard
# dashboard.save("crime_calendar.html")

alt.VConcatChart(...)